# Jacobian Analysis Test Notebook

This notebook tests the new Jacobian analysis tools from `occhio.analysis.jacobian`.

These tools measure how feature encoding directions vary across input contexts,
enabling detection of manifold structure in autoencoders.

**Key metrics:**
- **Angular variance**: 0 = linear (constant direction), ~1 = maximally nonlinear
- **Jacobian PCA**: Reveals dimensionality of the direction manifold
- **Direction vs context**: Identifies which features cause direction rotations

In [ ]:
import torch
import numpy as np
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from torch import Generator

from occhio import ToyModel, MLPAutoencoder
from occhio.autoencoder import TiedLinear, TiedLinearRelu
from occhio.distributions.sparse import SparseUniform
from occhio.analysis import (
    compute_feature_jacobians,
    angular_variance,
    jacobian_pca,
    direction_vs_context,
    compute_all_feature_jacobians,
)

## 1. Setup: Train Linear and Nonlinear Models

In [ ]:
# Configuration
N_FEATURES = 20
N_HIDDEN = 5
N_EPOCHS = 2000
P_ACTIVE = 0.2
BATCH_SIZE = 256
N_JACOBIAN_SAMPLES = 500


# Create distributions with same seed for fair comparison
def create_distribution():
    return SparseUniform(
        n_features=N_FEATURES,
        p_active=P_ACTIVE,
        generator=Generator().manual_seed(42),
    )


print(f"Config: {N_FEATURES} features -> {N_HIDDEN} hidden, p_active={P_ACTIVE}")

In [ ]:
# Train linear model (TiedLinear - no activation)
linear_model = ToyModel(
    distribution=create_distribution(),
    ae=TiedLinear(n_features=N_FEATURES, n_hidden=N_HIDDEN),
)
print("Training TiedLinear (linear encoder)...")
linear_losses, _ = linear_model.fit(
    n_epochs=N_EPOCHS, batch_size=BATCH_SIZE, track_losses=True
)
print(f"Final loss: {linear_losses[-1]:.6f}")

# Train ReLU model (TiedLinearRelu - piecewise linear)
relu_model = ToyModel(
    distribution=create_distribution(),
    ae=TiedLinearRelu(n_features=N_FEATURES, n_hidden=N_HIDDEN),
)
print("\nTraining TiedLinearRelu (ReLU decoder)...")
relu_losses, _ = relu_model.fit(
    n_epochs=N_EPOCHS, batch_size=BATCH_SIZE, track_losses=True
)
print(f"Final loss: {relu_losses[-1]:.6f}")

# Train MLP model (smooth nonlinearity)
mlp_model = ToyModel(
    distribution=create_distribution(),
    ae=MLPAutoencoder(
        n_features=N_FEATURES,
        n_hidden=N_HIDDEN,
        activation="gelu",
        decoder_activation="relu",  # Match SparseUniform non-negativity
    ),
)
print("\nTraining MLPAutoencoder (GELU encoder)...")
mlp_losses, _ = mlp_model.fit(
    n_epochs=N_EPOCHS, batch_size=BATCH_SIZE, track_losses=True
)
print(f"Final loss: {mlp_losses[-1]:.6f}")

## 2. Compute Feature Jacobians

For each model, compute Jacobians for a specific feature across many input contexts.
For linear encoders, all Jacobians should be identical. For nonlinear encoders, they may vary.

In [ ]:
# Generate test inputs where the target feature is active
FEATURE_IDX = 0  # Analyze feature 0

# Sample many inputs and filter for ones where feature is active
all_samples = create_distribution().sample(N_JACOBIAN_SAMPLES * 10)
active_mask = all_samples[:, FEATURE_IDX] > 0
test_inputs = all_samples[active_mask][:N_JACOBIAN_SAMPLES]

print(f"Using {test_inputs.shape[0]} samples where feature {FEATURE_IDX} is active")
print(
    f"Feature {FEATURE_IDX} mean activation: {test_inputs[:, FEATURE_IDX].mean():.4f}"
)

In [ ]:
# Compute Jacobians for each model
print("Computing Jacobians...")

with torch.no_grad():
    linear_jacs = compute_feature_jacobians(linear_model, FEATURE_IDX, test_inputs)
    relu_jacs = compute_feature_jacobians(relu_model, FEATURE_IDX, test_inputs)
    mlp_jacs = compute_feature_jacobians(mlp_model, FEATURE_IDX, test_inputs)

print(f"Jacobian shapes: {linear_jacs.shape}")
print(f"\nLinear model - first 3 Jacobians (should be identical):")
print(linear_jacs[:3].numpy().round(4))
print(f"\nMLP model - first 3 Jacobians (may differ):")
print(mlp_jacs[:3].numpy().round(4))

## 3. Angular Variance Analysis

Angular variance measures how much the encoding direction varies:
- **0**: All directions identical (perfectly linear)
- **~1**: Directions uniformly spread (maximally nonlinear)

In [ ]:
# Compute angular variance for each model
linear_av = angular_variance(linear_jacs)
relu_av = angular_variance(relu_jacs)
mlp_av = angular_variance(mlp_jacs)

print(f"Angular Variance for feature {FEATURE_IDX}:")
print(f"  TiedLinear:      {linear_av:.6f} (expected ~0)")
print(f"  TiedLinearRelu:  {relu_av:.6f}")
print(f"  MLPAutoencoder:  {mlp_av:.6f}")

In [ ]:
# Compute angular variance for ALL features
def compute_all_angular_variances(model, test_inputs):
    """Compute angular variance for every feature."""
    avs = []
    for feat_idx in range(model.n_features):
        jacs = compute_feature_jacobians(model, feat_idx, test_inputs)
        avs.append(angular_variance(jacs))
    return np.array(avs)


print("Computing angular variance for all features...")
linear_avs = compute_all_angular_variances(linear_model, test_inputs)
relu_avs = compute_all_angular_variances(relu_model, test_inputs)
mlp_avs = compute_all_angular_variances(mlp_model, test_inputs)

print(f"\nAngular Variance Summary (mean ± std):")
print(f"  TiedLinear:      {linear_avs.mean():.6f} ± {linear_avs.std():.6f}")
print(f"  TiedLinearRelu:  {relu_avs.mean():.6f} ± {relu_avs.std():.6f}")
print(f"  MLPAutoencoder:  {mlp_avs.mean():.6f} ± {mlp_avs.std():.6f}")

In [ ]:
# Visualize angular variance comparison
fig = make_subplots(
    rows=1,
    cols=2,
    subplot_titles=["Angular Variance by Feature", "Distribution Comparison"],
)

# Per-feature plot
features = list(range(N_FEATURES))
fig.add_trace(
    go.Scatter(x=features, y=linear_avs, name="TiedLinear", mode="markers+lines"),
    row=1,
    col=1,
)
fig.add_trace(
    go.Scatter(x=features, y=relu_avs, name="TiedLinearRelu", mode="markers+lines"),
    row=1,
    col=1,
)
fig.add_trace(
    go.Scatter(x=features, y=mlp_avs, name="MLPAutoencoder", mode="markers+lines"),
    row=1,
    col=1,
)

# Box plot comparison
fig.add_trace(go.Box(y=linear_avs, name="TiedLinear", boxpoints="all"), row=1, col=2)
fig.add_trace(go.Box(y=relu_avs, name="TiedLinearRelu", boxpoints="all"), row=1, col=2)
fig.add_trace(go.Box(y=mlp_avs, name="MLPAutoencoder", boxpoints="all"), row=1, col=2)

fig.update_layout(
    height=400,
    title_text="Angular Variance: Linear vs Nonlinear Encoders",
    showlegend=True,
)
fig.update_xaxes(title_text="Feature Index", row=1, col=1)
fig.update_yaxes(title_text="Angular Variance", row=1, col=1)
fig.update_yaxes(title_text="Angular Variance", row=1, col=2)
fig.show()

## 4. Jacobian PCA Analysis

PCA on Jacobian vectors reveals the intrinsic dimensionality of the direction manifold:
- **1 dominant eigenvalue**: Direction is fixed (linear)
- **Multiple significant eigenvalues**: Direction varies on a submanifold

In [ ]:
# PCA on Jacobians for feature 0
linear_eigs, linear_vecs = jacobian_pca(linear_jacs)
relu_eigs, relu_vecs = jacobian_pca(relu_jacs)
mlp_eigs, mlp_vecs = jacobian_pca(mlp_jacs)

print(f"PCA Eigenvalues for feature {FEATURE_IDX}:")
print(f"\nTiedLinear (should have ~0 variance, all directions same):")
print(f"  {linear_eigs.numpy().round(6)}")
print(f"\nTiedLinearRelu:")
print(f"  {relu_eigs.numpy().round(6)}")
print(f"\nMLPAutoencoder:")
print(f"  {mlp_eigs.numpy().round(6)}")

In [ ]:
# Plot eigenvalue spectra
fig = go.Figure()


# Normalize eigenvalues to show explained variance ratio
def explained_variance_ratio(eigs):
    total = eigs.sum().item()
    if total < 1e-10:
        return np.zeros(len(eigs))
    return (eigs / total).numpy()


components = list(range(1, N_HIDDEN + 1))
fig.add_trace(
    go.Bar(
        x=components,
        y=explained_variance_ratio(linear_eigs),
        name="TiedLinear",
        opacity=0.7,
    )
)
fig.add_trace(
    go.Bar(
        x=components,
        y=explained_variance_ratio(relu_eigs),
        name="TiedLinearRelu",
        opacity=0.7,
    )
)
fig.add_trace(
    go.Bar(
        x=components,
        y=explained_variance_ratio(mlp_eigs),
        name="MLPAutoencoder",
        opacity=0.7,
    )
)

fig.update_layout(
    title=f"PCA Eigenvalue Spectrum for Feature {FEATURE_IDX}",
    xaxis_title="Principal Component",
    yaxis_title="Explained Variance Ratio",
    barmode="group",
    height=400,
)
fig.show()

## 5. Direction vs Context Analysis

Identify which co-active features cause the encoding direction to rotate.

In [ ]:
# Analyze what causes direction changes for the MLP model
context_result = direction_vs_context(
    mlp_model,
    feature_idx=FEATURE_IDX,
    inputs=test_inputs,
    jacobians=mlp_jacs,
)

print(f"Direction vs Context Analysis for feature {FEATURE_IDX}:")
print(f"\nMost influential co-active features:")
for feat_idx, magnitude in context_result["most_influential"]:
    print(f"  Feature {feat_idx}: correlation magnitude = {magnitude:.4f}")

In [ ]:
# Visualize correlation magnitudes
corr_mags = context_result["correlation_magnitudes"].numpy()

fig = go.Figure(
    data=go.Bar(
        x=list(range(N_FEATURES)),
        y=corr_mags,
        marker_color=["red" if i == FEATURE_IDX else "blue" for i in range(N_FEATURES)],
    )
)

fig.update_layout(
    title=f"Influence of Each Feature on Feature {FEATURE_IDX}'s Encoding Direction",
    xaxis_title="Feature Index",
    yaxis_title="Correlation Magnitude",
    height=400,
)
fig.show()

## 6. 3D Visualization of Jacobian Directions

For low-dimensional hidden spaces (n_hidden=3), we can directly visualize
how the encoding direction varies on the unit sphere.

In [ ]:
# Train models with n_hidden=3 for visualization
N_HIDDEN_3D = 3

linear_3d = ToyModel(
    distribution=create_distribution(),
    ae=TiedLinear(n_features=N_FEATURES, n_hidden=N_HIDDEN_3D),
)
linear_3d.fit(n_epochs=N_EPOCHS, batch_size=BATCH_SIZE)

mlp_3d = ToyModel(
    distribution=create_distribution(),
    ae=MLPAutoencoder(
        n_features=N_FEATURES,
        n_hidden=N_HIDDEN_3D,
        activation="gelu",
        decoder_activation="relu",
    ),
)
mlp_3d.fit(n_epochs=N_EPOCHS, batch_size=BATCH_SIZE)

print("3D models trained.")

In [ ]:
# Compute and normalize Jacobians for 3D visualization
linear_jacs_3d = compute_feature_jacobians(linear_3d, FEATURE_IDX, test_inputs)
mlp_jacs_3d = compute_feature_jacobians(mlp_3d, FEATURE_IDX, test_inputs)

# Normalize to unit sphere
linear_normed = linear_jacs_3d / linear_jacs_3d.norm(dim=1, keepdim=True).clamp(
    min=1e-8
)
mlp_normed = mlp_jacs_3d / mlp_jacs_3d.norm(dim=1, keepdim=True).clamp(min=1e-8)

print(f"Linear AV: {angular_variance(linear_jacs_3d):.6f}")
print(f"MLP AV: {angular_variance(mlp_jacs_3d):.6f}")

In [ ]:
# 3D scatter plot of normalized Jacobian directions
fig = make_subplots(
    rows=1,
    cols=2,
    specs=[[{"type": "scatter3d"}, {"type": "scatter3d"}]],
    subplot_titles=["TiedLinear (should cluster)", "MLPAutoencoder (may spread)"],
)

# Linear model
fig.add_trace(
    go.Scatter3d(
        x=linear_normed[:, 0].detach().cpu().numpy(),
        y=linear_normed[:, 1].detach().cpu().numpy(),
        z=linear_normed[:, 2].detach().cpu().numpy(),
        mode="markers",
        marker=dict(size=3, color="blue", opacity=0.6),
        name="Linear",
    ),
    row=1,
    col=1,
)

# MLP model
# Color by activation of most influential feature
influential_feat = (
    context_result["most_influential"][0][0]
    if context_result["most_influential"]
    else 1
)
colors = test_inputs[:, influential_feat].numpy()

fig.add_trace(
    go.Scatter3d(
        x=mlp_normed[:, 0].detach().cpu().numpy(),
        y=mlp_normed[:, 1].detach().cpu().numpy(),
        z=mlp_normed[:, 2].detach().cpu().numpy(),
        mode="markers",
        marker=dict(
            size=3,
            color=colors,
            colorscale="Viridis",
            colorbar=dict(title=f"Feature {influential_feat}", x=1.0),
            opacity=0.6,
        ),
        name="MLP",
    ),
    row=1,
    col=2,
)

fig.update_layout(
    title=f"Jacobian Directions for Feature {FEATURE_IDX} on Unit Sphere",
    height=500,
    showlegend=False,
)
fig.show()

## Summary

The Jacobian analysis tools are working correctly:

1. **`compute_feature_jacobians`**: Computes ∂h/∂x_i for each input
2. **`angular_variance`**: Measures direction variability (0=linear, ~1=nonlinear)
3. **`jacobian_pca`**: Reveals dimensionality of the direction manifold
4. **`direction_vs_context`**: Identifies which features cause direction rotations

**Key findings from this test:**
- Linear models (TiedLinear) should show ~0 angular variance
- MLP models may show higher angular variance if they learn nonlinear encodings
- The 3D visualization shows whether directions cluster (linear) or spread (nonlinear)